# MAG Energy — Notebook d'expérimentation modèles & features

Ce notebook teste automatiquement plusieurs **groupes de features** et plusieurs **modèles** sur le `master_dataset` généré plus tôt.

Il est adapté à la structure observée dans ton notebook de préparation :
- `MONTH` = mois cible
- `DECISION_MONTH` = mois de décision
- `EID`, `PEAKID`
- `TARGET` = profitabilité binaire
- `PROFIT` = profit réel
- features historiques, `PSD`, `PSM`, calendrier

## Objectif
Comparer les combinaisons suivantes :
1. groupes de features
2. modèles de classification
3. modèles hybrides classification + régression
4. stratégie de sélection mensuelle avec contraintes business

## Rappel business
Le cas évalue à la fois :
- le **F1-score**
- le **profit total net**
- la capacité à sélectionner un nombre raisonnable d'opportunités par mois

La décision doit respecter une logique **temporelle sans fuite**, avec les contraintes du cutoff du 7 du mois M pour prédire M+1. fileciteturn2file2  
Le profit d'une opportunité est défini par `PR - C`, et la sélection doit rester entre **10 et 100 opportunités par mois**. fileciteturn2file4turn2file0

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    ExtraTreesClassifier,
    ExtraTreesRegressor,
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1) Chargement du dataset

Le notebook cherche d'abord un `master_dataset.parquet`, puis un `master_dataset.csv`.

Tu peux modifier `DATA_CANDIDATES` si ton fichier est ailleurs.

In [2]:
DATA_CANDIDATES = [
    Path("../data/master_dataset.parquet"),
    Path("../data/master_dataset.csv"),
    Path("./master_dataset.parquet"),
    Path("./master_dataset.csv"),
    Path("/mnt/data/master_dataset.parquet"),
    Path("/mnt/data/master_dataset.csv"),
]

data_path = None
for p in DATA_CANDIDATES:
    if p.exists():
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError(
        "Aucun master_dataset.parquet/csv trouvé. "
        "Ajoute le bon chemin dans DATA_CANDIDATES."
    )

print(f"Dataset trouvé : {data_path}")

if data_path.suffix == ".parquet":
    df = pd.read_parquet(data_path)
else:
    df = pd.read_csv(data_path)

print(df.shape)
df.head()

Dataset trouvé : ../data/master_dataset.parquet
(7998, 60)


,EID,MONTH,PEAKID,C,PR,PR_signed,PROFIT,TARGET,DECISION_MONTH,pr_lag1,pr_lag2,pr_lag3,c_lag1,profit_lag1,profit_lag2,target_lag1,pr_rolling3_mean,pr_rolling3_std,profit_rolling3_mean,profitable_count_3m,psd_nonzero_count,psd_abs_nonzero_mean,psd_abs_nonzero_std,psd_abs_sum,psd_signed_mean,psd_abs_max,activation_mean,activation_max,activation_nonzero_count,wind_abs_mean,solar_abs_mean,hydro_abs_mean,nonrenew_abs_mean,external_abs_mean,load_abs_mean,transoutage_abs_mean,psd_abs_s1_mean,psd_abs_s23_mean,psd_scenario_spread,psd_abs_early,psd_abs_late,psm_nonzero_count,psm_abs_nonzero_mean,psm_abs_nonzero_std,psm_abs_sum,psm_signed_mean,psm_abs_max,psm_activation_mean,psm_activation_max,psm_wind_abs_mean,psm_solar_abs_mean,psm_hydro_abs_mean,psm_nonrenew_abs_mean,psm_external_abs_mean,psm_abs_s1_mean,psm_abs_s23_mean,psm_scenario_spread,month_of_year,season,season_encoded
0,3,2020-09,0,15.829500,20.073800,-20.073800,4.244300,1,2020-08,43.918800,NaN,NaN,18.839701,25.079099,NaN,1.0,43.918800,NaN,25.079099,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0000,7.008313,63.573898,56.0,68.156061,0.0,22.101395,15.546763,4.114404,38.493536,1.222602,0.000000,0.000000,0.000000,0.000000,0.000000,149.0,3.247887,0.761124,483.935109,-3.247887,3.537100,53.641321,168.233307,363.496503,52.573589,229.375393,248.538675,114.353364,0.000000,0.630124,0.761124,9,fall,3
1,3,2020-10,0,67.694901,12.735000,-12.735000,-54.959901,0,2020-09,20.073800,43.9188,NaN,15.829500,4.244300,25.079099,1.0,31.996300,16.860961,14.661700,2.0,12.0,3.831058,2.007303,45.972700,-3.831058,10.2051,9.986848,100.000000,56.0,13.260586,0.0,4.433055,10.013485,1.196811,5.549011,0.397808,0.182234,0.319354,2.007303,0.000000,0.478882,39.0,5.006400,6.217145,195.249602,-5.006400,26.709999,39.061502,146.167999,59.207464,2.235974,28.684847,39.921203,14.196549,0.271703,0.113191,6.217145,10,fall,3
2,3,2020-12,0,66.708801,336.057098,-336.057098,269.348296,1,2020-11,12.735000,20.0738,43.9188,67.694901,-54.959901,4.244300,0.0,25.575867,16.303739,-8.545500,2.0,8.0,4.006975,0.510080,32.055801,0.431725,4.4387,7.266298,65.150902,56.0,38.602028,0.0,12.508409,11.833919,9.503315,6.028170,11.530199,0.000000,0.286213,4.289139,0.091719,0.265125,102.0,7.754970,10.486526,791.006989,-5.800212,50.935001,45.589009,185.843399,212.472694,4.033686,56.511475,151.466476,13.682372,1.029331,0.494272,11.693019,12,winter,0
3,3,2021-01,0,98.869499,237.243800,-237.243800,138.374301,1,2020-12,336.057098,12.7350,20.0738,66.708801,269.348296,-54.959901,1.0,122.955299,184.588046,72.877565,2.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0000,8.500296,65.941200,56.0,19.292411,0.0,3.757089,12.517613,2.956246,3.546621,1.745665,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,26.955167,54.235100,55.093534,0.000000,8.585233,1.903033,4.466167,0.000000,0.000000,0.000000,1,winter,0
4,3,2021-01,0,98.869499,237.243800,-237.243800,138.374301,1,2020-12,336.057098,12.7350,20.0738,66.708801,269.348296,-54.959901,1.0,122.955299,184.588046,72.877565,2.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0000,8.500296,65.941200,56.0,19.292411,0.0,3.757089,12.517613,2.956246,3.546621,1.745665,0.000000,0.000000,0.000000,0.000000,0.000000,27.0,12.406111,14.877385,334.965000,-12.406111,45.628700,34.295069,103.841400,168.626990,1.598633,49.083573,97.537761,12.034353,0.697059,0.047410,14.877385,1,winter,0


## 2) Standardisation des colonnes attendues

Le dataset issu de ton notebook contient normalement :
- `MONTH`
- `EID`
- `PEAKID`
- `TARGET`
- `PROFIT`

Ici on fait juste quelques vérifications et conversions.

In [3]:
DATE_COL = "MONTH"
DECISION_COL = "DECISION_MONTH"
ID_COLS = ["EID", "PEAKID"]
TARGET_CLASS = "TARGET"
TARGET_REG = "PROFIT"

required_cols = [DATE_COL, TARGET_CLASS, TARGET_REG] + ID_COLS
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Colonnes obligatoires manquantes: {missing_required}")

df = df.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL].astype(str))
if DECISION_COL in df.columns:
    # garde en string ou datetime selon l'existant, ce n'est pas bloquant
    pass

df[TARGET_CLASS] = df[TARGET_CLASS].astype(int)
df[TARGET_REG] = pd.to_numeric(df[TARGET_REG], errors="coerce")

print("Période :", df[DATE_COL].min(), "->", df[DATE_COL].max())
print("Taux de TARGET=1 :", round(df[TARGET_CLASS].mean() * 100, 2), "%")
print("Nb lignes :", len(df))

Période : 2020-02-01 00:00:00 -> 2023-12-01 00:00:00
Taux de TARGET=1 : 15.53 %
Nb lignes : 7998


## 3) Définition des groupes de features

Ces groupes reprennent directement les features construites dans ton notebook `master_dataset.ipynb`.

In [4]:
history_features = [
    "pr_lag1", "pr_lag2", "pr_lag3",
    "c_lag1",
    "profit_lag1", "profit_lag2",
    "target_lag1",
    "pr_rolling3_mean", "pr_rolling3_std",
    "profit_rolling3_mean",
    "profitable_count_3m",
]

psd_features = [
    "psd_nonzero_count",
    "psd_abs_nonzero_mean",
    "psd_abs_nonzero_std",
    "psd_abs_sum",
    "psd_signed_mean",
    "psd_abs_max",
    "activation_mean",
    "activation_max",
    "activation_nonzero_count",
    "wind_abs_mean",
    "solar_abs_mean",
    "hydro_abs_mean",
    "nonrenew_abs_mean",
    "external_abs_mean",
    "load_abs_mean",
    "transoutage_abs_mean",
    "psd_abs_s1_mean",
    "psd_abs_s23_mean",
    "psd_scenario_spread",
    "psd_abs_early",
    "psd_abs_late",
]

psm_features = [
    "psm_nonzero_count",
    "psm_abs_nonzero_mean",
    "psm_abs_nonzero_std",
    "psm_abs_sum",
    "psm_signed_mean",
    "psm_abs_max",
    "psm_activation_mean",
    "psm_activation_max",
    "psm_wind_abs_mean",
    "psm_solar_abs_mean",
    "psm_hydro_abs_mean",
    "psm_nonrenew_abs_mean",
    "psm_external_abs_mean",
    "psm_abs_s1_mean",
    "psm_abs_s23_mean",
    "psm_scenario_spread",
]

calendar_features = [
    "month_of_year",
    "season_encoded",
]

feature_sets_raw = {
    "history_only": history_features,
    "psd_only": psd_features,
    "psm_only": psm_features,
    "history_psd": history_features + psd_features,
    "history_psm": history_features + psm_features,
    "psd_psm": psd_features + psm_features,
    "history_psd_psm": history_features + psd_features + psm_features,
    "all_features": history_features + psd_features + psm_features + calendar_features,
}

In [5]:
# Garde uniquement les colonnes réellement présentes
feature_sets = {}
missing_by_group = {}

for name, cols in feature_sets_raw.items():
    present = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    feature_sets[name] = present
    missing_by_group[name] = missing

summary_features = pd.DataFrame({
    "feature_set": list(feature_sets.keys()),
    "n_features_present": [len(v) for v in feature_sets.values()],
    "n_features_missing": [len(missing_by_group[k]) for k in feature_sets.keys()],
})
summary_features

,feature_set,n_features_present,n_features_missing
0,history_only,11,0
1,psd_only,21,0
2,psm_only,16,0
3,history_psd,32,0
4,history_psm,27,0
5,psd_psm,37,0
6,history_psd_psm,48,0
7,all_features,50,0


In [6]:
for group_name in feature_sets:
    print("=" * 70)
    print(group_name)
    print(f"présentes: {len(feature_sets[group_name])}")
    if missing_by_group[group_name]:
        print("manquantes:", missing_by_group[group_name])
    else:
        print("manquantes: aucune")

history_only
présentes: 11
manquantes: aucune
psd_only
présentes: 21
manquantes: aucune
psm_only
présentes: 16
manquantes: aucune
history_psd
présentes: 32
manquantes: aucune
history_psm
présentes: 27
manquantes: aucune
psd_psm
présentes: 37
manquantes: aucune
history_psd_psm
présentes: 48
manquantes: aucune
all_features
présentes: 50
manquantes: aucune


## 4) Folds temporels walk-forward

On fait une validation temporelle par mois :
- train = tous les mois passés
- validation = mois courant

Pas de split aléatoire.

In [7]:
def build_monthly_folds(
    data: pd.DataFrame,
    date_col: str = DATE_COL,
    min_train_months: int = 12,
    max_folds: Optional[int] = None,
) -> List[Tuple[List[pd.Timestamp], pd.Timestamp]]:
    months = sorted(data[date_col].dt.to_period("M").unique())
    months = [m.to_timestamp() for m in months]

    folds = []
    for i in range(min_train_months, len(months)):
        train_months = months[:i]
        val_month = months[i]
        folds.append((train_months, val_month))

    if max_folds is not None:
        folds = folds[-max_folds:]

    return folds

folds = build_monthly_folds(df, date_col=DATE_COL, min_train_months=12, max_folds=None)
print("Nombre de folds :", len(folds))
print("Exemple :", folds[:3])

Nombre de folds : 35
Exemple : [([Timestamp('2020-02-01 00:00:00'), Timestamp('2020-03-01 00:00:00'), Timestamp('2020-04-01 00:00:00'), Timestamp('2020-05-01 00:00:00'), Timestamp('2020-06-01 00:00:00'), Timestamp('2020-07-01 00:00:00'), Timestamp('2020-08-01 00:00:00'), Timestamp('2020-09-01 00:00:00'), Timestamp('2020-10-01 00:00:00'), Timestamp('2020-11-01 00:00:00'), Timestamp('2020-12-01 00:00:00'), Timestamp('2021-01-01 00:00:00')], Timestamp('2021-02-01 00:00:00')), ([Timestamp('2020-02-01 00:00:00'), Timestamp('2020-03-01 00:00:00'), Timestamp('2020-04-01 00:00:00'), Timestamp('2020-05-01 00:00:00'), Timestamp('2020-06-01 00:00:00'), Timestamp('2020-07-01 00:00:00'), Timestamp('2020-08-01 00:00:00'), Timestamp('2020-09-01 00:00:00'), Timestamp('2020-10-01 00:00:00'), Timestamp('2020-11-01 00:00:00'), Timestamp('2020-12-01 00:00:00'), Timestamp('2021-01-01 00:00:00'), Timestamp('2021-02-01 00:00:00')], Timestamp('2021-03-01 00:00:00')), ([Timestamp('2020-02-01 00:00:00'), Timest

## 5) Fonctions business

Deux logiques de sélection sont incluses :
- `topk` : on prend les meilleurs scores du mois
- `threshold_then_cap` : seuil, puis on applique min/max

Dans ce cas, `topk` est souvent plus stable.

In [8]:
def select_top_k_monthly(
    month_df: pd.DataFrame,
    score_col: str,
    min_k: int = 10,
    max_k: int = 100,
    default_k: int = 30,
) -> pd.DataFrame:
    month_df = month_df.sort_values(score_col, ascending=False).copy()

    if len(month_df) == 0:
        return month_df.copy()

    k = min(default_k, len(month_df))
    k = max(k, min_k)
    k = min(k, max_k, len(month_df))

    return month_df.head(k).copy()


def select_threshold_then_cap(
    month_df: pd.DataFrame,
    score_col: str,
    threshold: float = 0.5,
    min_k: int = 10,
    max_k: int = 100,
) -> pd.DataFrame:
    month_df = month_df.sort_values(score_col, ascending=False).copy()
    selected = month_df[month_df[score_col] >= threshold].copy()

    if len(selected) < min_k:
        selected = month_df.head(min(min_k, len(month_df))).copy()
    elif len(selected) > max_k:
        selected = selected.head(max_k).copy()

    return selected


def compute_selection_metrics(
    selected_df: pd.DataFrame,
    target_class_col: str = TARGET_CLASS,
    target_reg_col: str = TARGET_REG,
) -> Dict[str, float]:
    if len(selected_df) == 0:
        return {
            "selected_count": 0,
            "selected_profit_sum": 0.0,
            "selected_profit_mean": 0.0,
            "selected_positive_rate": 0.0,
            "selected_precision": 0.0,
            "selected_recall_proxy": 0.0,
            "selected_f1_proxy": 0.0,
        }

    y_true = selected_df[target_class_col].values
    y_pred = np.ones(len(selected_df), dtype=int)

    return {
        "selected_count": int(len(selected_df)),
        "selected_profit_sum": float(selected_df[target_reg_col].sum()),
        "selected_profit_mean": float(selected_df[target_reg_col].mean()),
        "selected_positive_rate": float(selected_df[target_class_col].mean()),
        "selected_precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "selected_recall_proxy": float(recall_score(y_true, y_pred, zero_division=0)),
        "selected_f1_proxy": float(f1_score(y_true, y_pred, zero_division=0)),
    }

## 6) Modèles

On inclut des modèles robustes et simples à lancer :
- `logreg`
- `rf_clf`
- `extratrees_clf`

Pour le profit :
- `ridge_reg`
- `rf_reg`
- `extratrees_reg`

Tu pourras ensuite brancher XGBoost / LightGBM si vous les avez installés.

In [9]:
classification_models = {
    "logreg": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    "rf_clf": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
    "extratrees_clf": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=400,
            max_depth=10,
            min_samples_leaf=3,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
}

regression_models = {
    "ridge_reg": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]),
    "rf_reg": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
    "extratrees_reg": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=400,
            max_depth=10,
            min_samples_leaf=3,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
}

list(classification_models.keys()), list(regression_models.keys())

(['logreg', 'rf_clf', 'extratrees_clf'],
 ['ridge_reg', 'rf_reg', 'extratrees_reg'])

## 7) Fonction principale d'expérimentation

Modes de score disponibles :
- `proba` : proba de la classification
- `profit` : profit prédit
- `hybrid` : `proba * profit_prédit`

En général, le mode `hybrid` est très intéressant pour le cas business.

In [10]:
def _safe_predict_proba(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)[:, 1]
        return p

    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        raw = np.asarray(raw)
        denom = raw.max() - raw.min()
        if denom == 0:
            return np.full(len(raw), 0.5)
        return (raw - raw.min()) / (denom + 1e-12)

    pred = model.predict(X)
    pred = np.asarray(pred)
    denom = pred.max() - pred.min()
    if denom == 0:
        return np.full(len(pred), 0.5)
    return (pred - pred.min()) / (denom + 1e-12)


def _compute_global_metrics(y_true, y_pred, y_score, y_reg_true=None, y_reg_pred=None):
    out = {
        "full_f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "full_precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "full_recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "full_accuracy": float(accuracy_score(y_true, y_pred)),
    }
    try:
        out["full_auc"] = float(roc_auc_score(y_true, y_score))
    except Exception:
        out["full_auc"] = np.nan

    if y_reg_true is not None and y_reg_pred is not None:
        out["reg_mae"] = float(mean_absolute_error(y_reg_true, y_reg_pred))
        out["reg_rmse"] = float(np.sqrt(mean_squared_error(y_reg_true, y_reg_pred)))
    else:
        out["reg_mae"] = np.nan
        out["reg_rmse"] = np.nan

    return out


def run_experiment(
    data: pd.DataFrame,
    feature_cols: List[str],
    clf_model,
    reg_model=None,
    folds=None,
    date_col: str = DATE_COL,
    target_class_col: str = TARGET_CLASS,
    target_reg_col: str = TARGET_REG,
    min_k: int = 10,
    max_k: int = 100,
    default_k: int = 30,
    threshold: float = 0.5,
    selection_mode: str = "topk",
    score_mode: str = "proba",
) -> pd.DataFrame:

    if folds is None:
        raise ValueError("folds ne doit pas être None")

    results = []

    for fold_id, (train_months, val_month) in enumerate(folds):
        train_periods = [m.to_period("M") for m in train_months]
        val_period = val_month.to_period("M")

        train_mask = data[date_col].dt.to_period("M").isin(train_periods)
        val_mask = data[date_col].dt.to_period("M") == val_period

        train_df = data.loc[train_mask].copy()
        val_df = data.loc[val_mask].copy()

        if len(train_df) == 0 or len(val_df) == 0:
            continue

        X_train = train_df[feature_cols]
        y_train_cls = train_df[target_class_col]
        y_train_reg = train_df[target_reg_col]

        X_val = val_df[feature_cols]
        y_val_cls = val_df[target_class_col]
        y_val_reg = val_df[target_reg_col]

        clf = clone(clf_model)
        clf.fit(X_train, y_train_cls)

        val_df["score_proba"] = _safe_predict_proba(clf, X_val)
        val_df["pred_class"] = (val_df["score_proba"] >= threshold).astype(int)

        if reg_model is not None:
            reg = clone(reg_model)
            reg.fit(X_train, y_train_reg)
            val_df["score_profit_pred"] = reg.predict(X_val)
        else:
            val_df["score_profit_pred"] = val_df["score_proba"]

        if score_mode == "proba":
            val_df["final_score"] = val_df["score_proba"]
        elif score_mode == "profit":
            val_df["final_score"] = val_df["score_profit_pred"]
        elif score_mode == "hybrid":
            val_df["final_score"] = val_df["score_proba"] * val_df["score_profit_pred"]
        else:
            raise ValueError("score_mode doit être: 'proba', 'profit', ou 'hybrid'")

        global_metrics = _compute_global_metrics(
            y_true=y_val_cls,
            y_pred=val_df["pred_class"],
            y_score=val_df["score_proba"],
            y_reg_true=y_val_reg if reg_model is not None else None,
            y_reg_pred=val_df["score_profit_pred"] if reg_model is not None else None,
        )

        if selection_mode == "topk":
            selected_df = select_top_k_monthly(
                val_df,
                score_col="final_score",
                min_k=min_k,
                max_k=max_k,
                default_k=default_k,
            )
        elif selection_mode == "threshold_then_cap":
            selected_df = select_threshold_then_cap(
                val_df,
                score_col="final_score",
                threshold=threshold,
                min_k=min_k,
                max_k=max_k,
            )
        else:
            raise ValueError("selection_mode invalide")

        selection_metrics = compute_selection_metrics(
            selected_df,
            target_class_col=target_class_col,
            target_reg_col=target_reg_col,
        )

        results.append({
            "fold_id": fold_id,
            "val_month": val_month,
            "n_train": len(train_df),
            "n_val": len(val_df),
            "n_features": len(feature_cols),
            **global_metrics,
            **selection_metrics,
        })

    return pd.DataFrame(results)

## 8) Quick run

Commence par un sous-ensemble rapide.  
Ensuite tu pourras lancer la grille complète.

In [11]:
quick_feature_sets = {
    "history_only": feature_sets["history_only"],
    "history_psd": feature_sets["history_psd"],
    "history_psm": feature_sets["history_psm"],
    "all_features": feature_sets["all_features"],
}

quick_clf_models = {
    "logreg": classification_models["logreg"],
    "rf_clf": classification_models["rf_clf"],
}

quick_reg_models = {
    "ridge_reg": regression_models["ridge_reg"],
    "rf_reg": regression_models["rf_reg"],
}

In [12]:
all_results = []

for feature_set_name, feature_cols in quick_feature_sets.items():
    if len(feature_cols) == 0:
        print(f"[SKIP] {feature_set_name} vide")
        continue

    for clf_name, clf_model in quick_clf_models.items():
        # classification seule
        res = run_experiment(
            data=df,
            feature_cols=feature_cols,
            clf_model=clf_model,
            reg_model=None,
            folds=folds,
            min_k=10,
            max_k=100,
            default_k=30,
            threshold=0.5,
            selection_mode="topk",
            score_mode="proba",
        )
        if len(res) > 0:
            res["feature_set"] = feature_set_name
            res["clf_model"] = clf_name
            res["reg_model"] = None
            res["score_mode"] = "proba"
            all_results.append(res)

        # hybride
        for reg_name, reg_model in quick_reg_models.items():
            res = run_experiment(
                data=df,
                feature_cols=feature_cols,
                clf_model=clf_model,
                reg_model=reg_model,
                folds=folds,
                min_k=10,
                max_k=100,
                default_k=30,
                threshold=0.5,
                selection_mode="topk",
                score_mode="hybrid",
            )
            if len(res) > 0:
                res["feature_set"] = feature_set_name
                res["clf_model"] = clf_name
                res["reg_model"] = reg_name
                res["score_mode"] = "hybrid"
                all_results.append(res)

results_df = pd.concat(all_results, ignore_index=True)
print(results_df.shape)


(840, 23)


,fold_id,val_month,n_train,n_val,n_features,full_f1,full_precision,full_recall,full_accuracy,full_auc,reg_mae,reg_rmse,selected_count,selected_profit_sum,selected_profit_mean,selected_positive_rate,selected_precision,selected_recall_proxy,selected_f1_proxy,feature_set,clf_model,reg_model,score_mode
0,0,2021-02-01,1392,209,11,0.297872,0.197183,0.608696,0.684211,0.600514,NaN,NaN,30,-49229.206027,-1640.973534,0.300000,0.300000,1.0,0.461538,history_only,logreg,None,proba
1,1,2021-03-01,1601,148,11,0.253521,0.176471,0.450000,0.641892,0.630469,NaN,NaN,30,-74298.165247,-2476.605508,0.166667,0.166667,1.0,0.285714,history_only,logreg,None,proba
2,2,2021-04-01,1749,138,11,0.405797,0.304348,0.608696,0.702899,0.623440,NaN,NaN,30,-106782.963089,-3559.432103,0.233333,0.233333,1.0,0.378378,history_only,logreg,None,proba
3,3,2021-05-01,1887,133,11,0.324324,0.244898,0.480000,0.624060,0.605926,NaN,NaN,30,-102157.173042,-3405.239101,0.333333,0.333333,1.0,0.500000,history_only,logreg,None,proba
4,4,2021-06-01,2020,132,11,0.463768,0.380952,0.592593,0.719697,0.691005,NaN,NaN,30,-26725.258171,-890.841939,0.433333,0.433333,1.0,0.604651,history_only,logreg,None,proba


In [14]:
results_df.to_csv("feature_testing_results.csv", index=False)

In [22]:
summary = (
    results_df
    .groupby(["feature_set","clf_model","reg_model","score_mode"])
    .agg({
        "selected_profit_sum": "mean",
        "selected_f1_proxy": "mean",
        "selected_count": "mean",
    })
    .sort_values("selected_profit_sum", ascending=False)
)

summary.head(10)

selected_profit_sum  selected_f1_proxy  selected_count
feature_set  clf_model reg_model score_mode                                                        
history_psm  logreg    rf_reg    hybrid              9719.405921           0.263001            30.0
all_features logreg    rf_reg    hybrid              8835.498424           0.264503            30.0
history_psd  logreg    ridge_reg hybrid              6286.338020           0.439953            30.0
             rf_clf    ridge_reg hybrid              4736.777495           0.439889            30.0
all_features rf_clf    rf_reg    hybrid              4616.438434           0.251652            30.0
history_only logreg    ridge_reg hybrid               314.690213           0.457049            30.0
             rf_clf    ridge_reg hybrid              -233.689744           0.440945            30.0
history_psm  rf_clf    rf_reg    hybrid             -1912.151826           0.246294            30.0
history_psd  logreg    rf_reg    hybrid             -2324.534424           0.260157            30.0
history_only logreg    rf_reg    hybrid             -5545.694666           0.305413            30.0

## 9) Résumé agrégé

In [16]:
summary = (
    results_df
    .groupby(["feature_set", "clf_model", "reg_model", "score_mode"], dropna=False)
    .agg(
        folds=("fold_id", "count"),
        n_features=("n_features", "mean"),
        full_f1_mean=("full_f1", "mean"),
        full_precision_mean=("full_precision", "mean"),
        full_recall_mean=("full_recall", "mean"),
        full_auc_mean=("full_auc", "mean"),
        selected_count_mean=("selected_count", "mean"),
        selected_profit_total=("selected_profit_sum", "sum"),
        selected_profit_mean=("selected_profit_sum", "mean"),
        selected_avg_profit_mean=("selected_profit_mean", "mean"),
        selected_precision_mean=("selected_precision", "mean"),
        selected_positive_rate_mean=("selected_positive_rate", "mean"),
        selected_f1_proxy_mean=("selected_f1_proxy", "mean"),
    )
    .reset_index()
)

summary = summary.sort_values(
    ["selected_profit_total", "selected_f1_proxy_mean", "full_f1_mean"],
    ascending=[False, False, False]
).reset_index(drop=True)

summary.head(20)

,feature_set,clf_model,reg_model,score_mode,folds,n_features,full_f1_mean,full_precision_mean,full_recall_mean,full_auc_mean,selected_count_mean,selected_profit_total,selected_profit_mean,selected_avg_profit_mean,selected_precision_mean,selected_positive_rate_mean,selected_f1_proxy_mean
0,all_features,rf_clf,NaN,proba,35,50.0,0.368416,0.341199,0.429033,0.740745,30.0,491320.973702,14037.742106,467.924737,0.357143,0.357143,0.512077
1,history_psm,logreg,rf_reg,hybrid,35,27.0,0.371633,0.289050,0.551980,0.701212,30.0,340179.207242,9719.405921,323.980197,0.161905,0.161905,0.263001
2,all_features,logreg,rf_reg,hybrid,35,50.0,0.368035,0.287141,0.547044,0.700670,30.0,309242.444844,8835.498424,294.516614,0.161905,0.161905,0.264503
3,history_psd,logreg,ridge_reg,hybrid,35,32.0,0.356512,0.269250,0.563590,0.681199,30.0,220021.830696,6286.338020,209.544601,0.295238,0.295238,0.439953
4,history_psd,rf_clf,ridge_reg,hybrid,35,32.0,0.369790,0.335529,0.473139,0.731365,30.0,165787.212327,4736.777495,157.892583,0.295238,0.295238,0.439889
5,all_features,rf_clf,rf_reg,hybrid,35,50.0,0.368416,0.341199,0.429033,0.740745,30.0,161575.345187,4616.438434,153.881281,0.152381,0.152381,0.251652
6,history_only,logreg,ridge_reg,hybrid,35,11.0,0.360183,0.271925,0.568874,0.665029,30.0,11014.157443,314.690213,10.489674,0.308571,0.308571,0.457049
7,history_only,rf_clf,ridge_reg,hybrid,35,11.0,0.349163,0.290399,0.489130,0.704226,30.0,-8179.141034,-233.689744,-7.789658,0.296190,0.296190,0.440945
8,history_psm,rf_clf,rf_reg,hybrid,35,27.0,0.370961,0.329413,0.448252,0.734400,30.0,-66925.313912,-1912.151826,-63.738394,0.150476,0.150476,0.246294
9,history_psd,logreg,rf_reg,hybrid,35,32.0,0.356512,0.269250,0.563590,0.681199,30.0,-81358.704826,-2324.534424,-77.484481,0.159048,0.159048,0.260157


In [17]:
print("=== TOP 10 PROFIT TOTAL ===")
display(summary.head(10))

print("\n=== TOP 10 PAR F1 GLOBAL ===")
display(summary.sort_values("full_f1_mean", ascending=False).head(10))

=== TOP 10 PROFIT TOTAL ===


,feature_set,clf_model,reg_model,score_mode,folds,n_features,full_f1_mean,full_precision_mean,full_recall_mean,full_auc_mean,selected_count_mean,selected_profit_total,selected_profit_mean,selected_avg_profit_mean,selected_precision_mean,selected_positive_rate_mean,selected_f1_proxy_mean
0,all_features,rf_clf,NaN,proba,35,50.0,0.368416,0.341199,0.429033,0.740745,30.0,491320.973702,14037.742106,467.924737,0.357143,0.357143,0.512077
1,history_psm,logreg,rf_reg,hybrid,35,27.0,0.371633,0.289050,0.551980,0.701212,30.0,340179.207242,9719.405921,323.980197,0.161905,0.161905,0.263001
2,all_features,logreg,rf_reg,hybrid,35,50.0,0.368035,0.287141,0.547044,0.700670,30.0,309242.444844,8835.498424,294.516614,0.161905,0.161905,0.264503
3,history_psd,logreg,ridge_reg,hybrid,35,32.0,0.356512,0.269250,0.563590,0.681199,30.0,220021.830696,6286.338020,209.544601,0.295238,0.295238,0.439953
4,history_psd,rf_clf,ridge_reg,hybrid,35,32.0,0.369790,0.335529,0.473139,0.731365,30.0,165787.212327,4736.777495,157.892583,0.295238,0.295238,0.439889
5,all_features,rf_clf,rf_reg,hybrid,35,50.0,0.368416,0.341199,0.429033,0.740745,30.0,161575.345187,4616.438434,153.881281,0.152381,0.152381,0.251652
6,history_only,logreg,ridge_reg,hybrid,35,11.0,0.360183,0.271925,0.568874,0.665029,30.0,11014.157443,314.690213,10.489674,0.308571,0.308571,0.457049
7,history_only,rf_clf,ridge_reg,hybrid,35,11.0,0.349163,0.290399,0.489130,0.704226,30.0,-8179.141034,-233.689744,-7.789658,0.296190,0.296190,0.440945
8,history_psm,rf_clf,rf_reg,hybrid,35,27.0,0.370961,0.329413,0.448252,0.734400,30.0,-66925.313912,-1912.151826,-63.738394,0.150476,0.150476,0.246294
9,history_psd,logreg,rf_reg,hybrid,35,32.0,0.356512,0.269250,0.563590,0.681199,30.0,-81358.704826,-2324.534424,-77.484481,0.159048,0.159048,0.260157



=== TOP 10 PAR F1 GLOBAL ===


,feature_set,clf_model,reg_model,score_mode,folds,n_features,full_f1_mean,full_precision_mean,full_recall_mean,full_auc_mean,selected_count_mean,selected_profit_total,selected_profit_mean,selected_avg_profit_mean,selected_precision_mean,selected_positive_rate_mean,selected_f1_proxy_mean
12,history_psm,logreg,ridge_reg,hybrid,35,27.0,0.371633,0.289050,0.551980,0.701212,30.0,-2.166614e+05,-6190.324947,-206.344165,0.284762,0.284762,0.426404
1,history_psm,logreg,rf_reg,hybrid,35,27.0,0.371633,0.289050,0.551980,0.701212,30.0,3.401792e+05,9719.405921,323.980197,0.161905,0.161905,0.263001
23,history_psm,logreg,NaN,proba,35,27.0,0.371633,0.289050,0.551980,0.701212,30.0,-1.304192e+06,-37262.624738,-1242.087491,0.337143,0.337143,0.490621
18,history_psm,rf_clf,NaN,proba,35,27.0,0.370961,0.329413,0.448252,0.734400,30.0,-4.689701e+05,-13399.144436,-446.638148,0.349524,0.349524,0.500833
8,history_psm,rf_clf,rf_reg,hybrid,35,27.0,0.370961,0.329413,0.448252,0.734400,30.0,-6.692531e+04,-1912.151826,-63.738394,0.150476,0.150476,0.246294
14,history_psm,rf_clf,ridge_reg,hybrid,35,27.0,0.370961,0.329413,0.448252,0.734400,30.0,-3.359745e+05,-9599.272423,-319.975747,0.271429,0.271429,0.407157
10,history_psd,rf_clf,NaN,proba,35,32.0,0.369790,0.335529,0.473139,0.731365,30.0,-1.721006e+05,-4917.159658,-163.905322,0.351429,0.351429,0.503118
4,history_psd,rf_clf,ridge_reg,hybrid,35,32.0,0.369790,0.335529,0.473139,0.731365,30.0,1.657872e+05,4736.777495,157.892583,0.295238,0.295238,0.439889
16,history_psd,rf_clf,rf_reg,hybrid,35,32.0,0.369790,0.335529,0.473139,0.731365,30.0,-3.524637e+05,-10070.392631,-335.679754,0.142857,0.142857,0.239577
0,all_features,rf_clf,NaN,proba,35,50.0,0.368416,0.341199,0.429033,0.740745,30.0,4.913210e+05,14037.742106,467.924737,0.357143,0.357143,0.512077


## 10) Grille complète

Quand le quick run marche, tu peux lancer toute la comparaison ici.  
Attention : ça peut prendre plus de temps.

In [19]:
RUN_FULL_GRID = False

In [20]:
full_results_df = None
full_summary = None

if RUN_FULL_GRID:
    all_results_full = []

    for feature_set_name, feature_cols in feature_sets.items():
        if len(feature_cols) == 0:
            print(f"[SKIP] {feature_set_name} vide")
            continue

        for clf_name, clf_model in classification_models.items():
            res = run_experiment(
                data=df,
                feature_cols=feature_cols,
                clf_model=clf_model,
                reg_model=None,
                folds=folds,
                min_k=10,
                max_k=100,
                default_k=30,
                threshold=0.5,
                selection_mode="topk",
                score_mode="proba",
            )
            if len(res) > 0:
                res["feature_set"] = feature_set_name
                res["clf_model"] = clf_name
                res["reg_model"] = None
                res["score_mode"] = "proba"
                all_results_full.append(res)

            for reg_name, reg_model in regression_models.items():
                res = run_experiment(
                    data=df,
                    feature_cols=feature_cols,
                    clf_model=clf_model,
                    reg_model=reg_model,
                    folds=folds,
                    min_k=10,
                    max_k=100,
                    default_k=30,
                    threshold=0.5,
                    selection_mode="topk",
                    score_mode="hybrid",
                )
                if len(res) > 0:
                    res["feature_set"] = feature_set_name
                    res["clf_model"] = clf_name
                    res["reg_model"] = reg_name
                    res["score_mode"] = "hybrid"
                    all_results_full.append(res)

    full_results_df = pd.concat(all_results_full, ignore_index=True)

    full_summary = (
        full_results_df
        .groupby(["feature_set", "clf_model", "reg_model", "score_mode"], dropna=False)
        .agg(
            folds=("fold_id", "count"),
            n_features=("n_features", "mean"),
            full_f1_mean=("full_f1", "mean"),
            full_precision_mean=("full_precision", "mean"),
            full_recall_mean=("full_recall", "mean"),
            full_auc_mean=("full_auc", "mean"),
            selected_count_mean=("selected_count", "mean"),
            selected_profit_total=("selected_profit_sum", "sum"),
            selected_profit_mean=("selected_profit_sum", "mean"),
            selected_avg_profit_mean=("selected_profit_mean", "mean"),
            selected_precision_mean=("selected_precision", "mean"),
            selected_positive_rate_mean=("selected_positive_rate", "mean"),
            selected_f1_proxy_mean=("selected_f1_proxy", "mean"),
        )
        .reset_index()
        .sort_values(["selected_profit_total", "selected_f1_proxy_mean"], ascending=[False, False])
        .reset_index(drop=True)
    )

    display(full_summary.head(25))
else:
    print("RUN_FULL_GRID = False, donc la grille complète n'est pas lancée.")

RUN_FULL_GRID = False, donc la grille complète n'est pas lancée.


## 11) Importance des features

Cette cellule prend le meilleur modèle du quick run et calcule une importance si le modèle final l'expose.  
Sinon, elle essaie une permutation importance simple.

In [23]:
from sklearn.inspection import permutation_importance

# Support both the quick-summary (reset index) and the grouped summary (index contains the group keys).
summary_for_best = summary.reset_index() if "feature_set" not in summary.columns else summary
best_row = summary_for_best.iloc[0]
best_feature_set = best_row["feature_set"]
best_clf_name = best_row["clf_model"]

best_features = quick_feature_sets[best_feature_set]
best_model = clone(quick_clf_models[best_clf_name])

X_all = df[best_features]
y_all = df[TARGET_CLASS]

best_model.fit(X_all, y_all)

importance_df = None

# Cas arbres
final_estimator = best_model.named_steps.get("model", best_model)
if hasattr(final_estimator, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": best_features,
        "importance": final_estimator.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)
else:
    perm = permutation_importance(
        best_model,
        X_all,
        y_all,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    importance_df = pd.DataFrame({
        "feature": best_features,
        "importance": perm.importances_mean,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

importance_df.head(25)

,feature,importance
0,psm_scenario_spread,0.031858
1,profitable_count_3m,0.029157
2,target_lag1,0.020055
3,psm_abs_nonzero_std,0.016879
4,pr_rolling3_mean,0.004851
5,profit_lag1,0.004126
6,pr_lag1,0.003401
7,psm_nonzero_count,0.003376
8,psm_nonrenew_abs_mean,0.002251
9,psm_abs_sum,0.000900


## 12) Ablation study

On enlève des blocs depuis `all_features` pour voir ce qui fait vraiment baisser les performances.

In [24]:
def ablation_from_all(remove_blocks: List[str]) -> List[str]:
    block_map = {
        "history": history_features,
        "psd": psd_features,
        "psm": psm_features,
        "calendar": calendar_features,
    }
    base = feature_sets["all_features"].copy()
    to_remove = set()
    for b in remove_blocks:
        to_remove.update([c for c in block_map[b] if c in base])
    return [c for c in base if c not in to_remove]

ablation_sets = {
    "all_minus_history": ablation_from_all(["history"]),
    "all_minus_psd": ablation_from_all(["psd"]),
    "all_minus_psm": ablation_from_all(["psm"]),
    "all_minus_calendar": ablation_from_all(["calendar"]),
}

In [25]:
ablation_results = []

ablation_clf = classification_models["rf_clf"]
ablation_reg = regression_models["rf_reg"]

for name, feat_cols in ablation_sets.items():
    if len(feat_cols) == 0:
        continue

    res = run_experiment(
        data=df,
        feature_cols=feat_cols,
        clf_model=ablation_clf,
        reg_model=ablation_reg,
        folds=folds,
        min_k=10,
        max_k=100,
        default_k=30,
        threshold=0.5,
        selection_mode="topk",
        score_mode="hybrid",
    )
    if len(res) > 0:
        res["ablation_name"] = name
        ablation_results.append(res)

if ablation_results:
    ablation_df = pd.concat(ablation_results, ignore_index=True)
    ablation_summary = (
        ablation_df.groupby("ablation_name")
        .agg(
            full_f1_mean=("full_f1", "mean"),
            selected_profit_total=("selected_profit_sum", "sum"),
            selected_precision_mean=("selected_precision", "mean"),
            selected_positive_rate_mean=("selected_positive_rate", "mean"),
        )
        .sort_values("selected_profit_total", ascending=False)
    )
    display(ablation_summary)
else:
    print("Aucun résultat d'ablation.")

,full_f1_mean,selected_profit_total,selected_precision_mean,selected_positive_rate_mean
ablation_name,,,,
all_minus_calendar,0.379478,127826.542339,0.145714,0.145714
all_minus_psd,0.367623,104694.367767,0.156190,0.156190
all_minus_psm,0.366346,-431129.203189,0.142857,0.142857
all_minus_history,0.341690,-721121.508737,0.143810,0.143810


## 13) Génération d'une sélection mensuelle finale

Cette partie permet de produire un `opportunities.csv` pour une stratégie choisie.
Adapte `FINAL_*` si tu veux utiliser un autre modèle/groupe.

In [ ]:
FINAL_FEATURE_SET = summary.iloc[0]["feature_set"]
FINAL_CLF = summary.iloc[0]["clf_model"]
FINAL_REG = summary.iloc[0]["reg_model"]
FINAL_SCORE_MODE = summary.iloc[0]["score_mode"]

print("Feature set final :", FINAL_FEATURE_SET)
print("Modèle clf final  :", FINAL_CLF)
print("Modèle reg final  :", FINAL_REG)
print("Score mode        :", FINAL_SCORE_MODE)

In [ ]:
def fit_and_score_month(
    train_df: pd.DataFrame,
    predict_df: pd.DataFrame,
    feature_cols: List[str],
    clf_model,
    reg_model=None,
    threshold: float = 0.5,
    score_mode: str = "hybrid",
):
    X_train = train_df[feature_cols]
    y_train_cls = train_df[TARGET_CLASS]
    y_train_reg = train_df[TARGET_REG]

    X_pred = predict_df[feature_cols]

    clf = clone(clf_model)
    clf.fit(X_train, y_train_cls)
    predict_df = predict_df.copy()
    predict_df["score_proba"] = _safe_predict_proba(clf, X_pred)

    if reg_model is not None:
        reg = clone(reg_model)
        reg.fit(X_train, y_train_reg)
        predict_df["score_profit_pred"] = reg.predict(X_pred)
    else:
        predict_df["score_profit_pred"] = predict_df["score_proba"]

    if score_mode == "proba":
        predict_df["final_score"] = predict_df["score_proba"]
    elif score_mode == "profit":
        predict_df["final_score"] = predict_df["score_profit_pred"]
    elif score_mode == "hybrid":
        predict_df["final_score"] = predict_df["score_proba"] * predict_df["score_profit_pred"]
    else:
        raise ValueError("score_mode invalide")

    predict_df["pred_class"] = (predict_df["score_proba"] >= threshold).astype(int)
    return predict_df

In [ ]:
# Refit mois par mois sur toute l'historique précédent, puis sélection mensuelle
final_feature_cols = feature_sets[FINAL_FEATURE_SET]
final_clf_model = classification_models[FINAL_CLF]
final_reg_model = None if pd.isna(FINAL_REG) else regression_models[FINAL_REG]

months = sorted(df[DATE_COL].dt.to_period("M").unique())
months = [m.to_timestamp() for m in months]

monthly_selections = []

for i in range(12, len(months)):
    train_months = months[:i]
    target_month = months[i]

    train_mask = df[DATE_COL].dt.to_period("M").isin([m.to_period("M") for m in train_months])
    pred_mask = df[DATE_COL].dt.to_period("M") == target_month.to_period("M")

    train_df = df.loc[train_mask].copy()
    pred_df = df.loc[pred_mask].copy()

    if len(train_df) == 0 or len(pred_df) == 0:
        continue

    scored = fit_and_score_month(
        train_df=train_df,
        predict_df=pred_df,
        feature_cols=final_feature_cols,
        clf_model=final_clf_model,
        reg_model=final_reg_model,
        threshold=0.5,
        score_mode=FINAL_SCORE_MODE,
    )

    selected = select_top_k_monthly(
        scored,
        score_col="final_score",
        min_k=10,
        max_k=100,
        default_k=30,
    )

    monthly_selections.append(selected)

if monthly_selections:
    selected_all = pd.concat(monthly_selections, ignore_index=True)
else:
    selected_all = pd.DataFrame()

selected_all.head()

In [ ]:
if len(selected_all) > 0:
    output = selected_all.copy()

    # Format attendu dans le case :
    # TARGET_MONTH, PEAK_TYPE, EID
    output["TARGET_MONTH"] = output[DATE_COL].dt.strftime("%Y-%m")
    output["PEAK_TYPE"] = output["PEAKID"].map({1: "ON", 0: "OFF"}).fillna(output["PEAKID"].astype(str))

    opportunities = (
        output[["TARGET_MONTH", "PEAK_TYPE", "EID"]]
        .drop_duplicates()
        .sort_values(["TARGET_MONTH", "PEAK_TYPE", "EID"])
        .reset_index(drop=True)
    )

    opportunities.to_csv("opportunities.csv", index=False)
    print("opportunities.csv exporté")
    display(opportunities.head(20))
else:
    print("Aucune sélection générée.")

## 14) Exports

Le notebook exporte :
- résultats fold par fold
- résumé agrégé
- importances
- opportunités finales

In [ ]:
results_df.to_csv("experiment_fold_results_quick.csv", index=False)
summary.to_csv("experiment_summary_quick.csv", index=False)
importance_df.to_csv("feature_importances_best_model.csv", index=False)

if full_results_df is not None:
    full_results_df.to_csv("experiment_fold_results_full.csv", index=False)
if full_summary is not None:
    full_summary.to_csv("experiment_summary_full.csv", index=False)

print("Exports terminés.")

## 15) Conseils d'interprétation

Quand vous comparez les modèles, regardez surtout :
- `selected_profit_total`
- `selected_f1_proxy_mean`
- `selected_precision_mean`
- `selected_count_mean`

Un bon modèle pour le case est souvent celui qui :
- garde un **profit total élevé**
- sélectionne des opportunités **vraiment rentables**
- reste **stable dans le temps**
- respecte naturellement la contrainte des **10 à 100 opportunités par mois**. fileciteturn2file4turn2file3